# 02 — Tuzların Çıkarılması, Tekrarlı SMILES Birleştirme ve Lipinski Filtresi

**Hedef:** Notebook 01 çıktısını kimyasal açıdan standardize etmek; geçersiz SMILES'ları belirlemek; tuzları/karşı iyonları uzaklaştırmak; aynı parent yapıya dönüşen kayıtları birleştirmek ve Lipinski Rule of Five filtresini uygulamak.

### Bu notebookta öğreneceklerimiz
- RDKit ile SMILES doğrulama
- `FragmentParent` ile tuz/karşı iyon temizliği
- Canonical parent SMILES üretimi
- Tekrarlı aktivite kayıtlarını medyan ile birleştirme
- Lipinski kuralları ve filtreleme raporu

### Ders planındaki yeri
**Ders 1 / Bölüm B:** Kimyasal veri kürasyonu.

## Workshop dosya yapısı

Bu seri çalıştırıldığında Google Drive altında aşağıdaki yapı oluşur:

```text
MyDrive/
└── CHEMBL206_workshop/
    ├── 01_data/
    ├── 02_cleaned/
    ├── 03_features/
    ├── 04_models/
    └── 05_reports/
```

> **Çalıştırma sırası:** Notebook 01 → 02 → 03 → 04 → 05  
> Her kod hücresinin sonunda bir **CHECKPOINT** mesajı vardır. Bir hücre hata verirse sonraki hücreye geçmeyiniz.

## Bilimsel not: “Tekrarlı SMILES'ı silmek” yerine neden birleştiriyoruz?

Aynı parent molekül için birden fazla IC50 ölçümü bulunabilir. Rastgele bir satırı tutmak deneysel bilgiyi kaybettirir. Bu notebook varsayılan olarak:

- pIC50 değerlerinin **medyanını** alır,
- tekrar sayısını `n_records` olarak saklar,
- ölçümler arası değişkenliği `pIC50_std` ile raporlar.

Bu yaklaşım özellikle uç değerlerin etkisini azaltır.

## Lipinski Rule of Five

Bu notebook aşağıdaki dört eşik için ihlal sayısı hesaplar:

- Molekül ağırlığı ≤ 500 Da
- LogP ≤ 5
- H-bağı vericisi ≤ 5
- H-bağı alıcısı ≤ 10

`Lipinski_violations == 0` olan moleküller workshop modelleme setine alınır.

In [ ]:
# Google Drive bağlantısını kuruyoruz.
from google.colab import drive

# Drive'ı standart Colab yoluna bağlıyoruz.
drive.mount("/content/drive")

print("✅ CHECKPOINT 02.1: Google Drive bağlantısı kuruldu.")

In [ ]:
# RDKit ve görselleştirme bağımlılıklarını Colab ortamına kuruyoruz.
# - rdkit: molekül okuma, standardizasyon ve fizikokimyasal özellikler
# - tqdm: ilerleme çubuğu
!pip -q install rdkit tqdm

print("✅ CHECKPOINT 02.2: RDKit ve yardımcı paketlerin kurulumu tamamlandı.")

In [ ]:
# Dosya yolu işlemleri için pathlib kullanıyoruz.
from pathlib import Path

# Tablo işlemleri için pandas kullanıyoruz.
import pandas as pd

# Sayısal işlemler için numpy kullanıyoruz.
import numpy as np

# İlerleme çubuğu için tqdm kullanıyoruz.
from tqdm.auto import tqdm

# RDKit temel kimya modülünü içe aktarıyoruz.
from rdkit import Chem

# Moleküler standardizasyon araçlarını içe aktarıyoruz.
from rdkit.Chem.MolStandardize import rdMolStandardize

# Lipinski özellikleri için gerekli descriptor modüllerini içe aktarıyoruz.
from rdkit.Chem import Descriptors, Crippen, Lipinski

# Molekül görselleri için Draw modülünü içe aktarıyoruz.
from rdkit.Chem import Draw

# Notebook görüntüleme yardımcısını içe aktarıyoruz.
from IPython.display import display

# tqdm ilerleme çubuğunu pandas apply ile kullanıma açıyoruz.
tqdm.pandas()

# Proje klasörlerini tanımlıyoruz.
PROJECT_ROOT = Path("/content/drive/MyDrive/CHEMBL206_workshop")
DATA_DIR = PROJECT_ROOT / "01_data"
CLEAN_DIR = PROJECT_ROOT / "02_cleaned"

# Çıktı klasörünü oluşturuyoruz.
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Notebook 01 çıktısını giriş dosyası olarak tanımlıyoruz.
INPUT_CSV = DATA_DIR / "CHEMBL206_IC50_workshop_base.csv"

# Giriş dosyasının varlığını kontrol ediyoruz.
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        "Notebook 01 çıktısı bulunamadı. Önce Notebook 01'i çalıştırınız:\n"
        f"{INPUT_CSV}"
    )

print("Giriş CSV:", INPUT_CSV)
print("Çıktı klasörü:", CLEAN_DIR)
print("✅ CHECKPOINT 02.3: Klasörler ve giriş dosyası hazır.")

In [ ]:
# Notebook 01 çıktısını okuyoruz.
df = pd.read_csv(INPUT_CSV, low_memory=False)

# Zorunlu sütunların bulunup bulunmadığını kontrol ediyoruz.
required_columns = {"molecule_id", "smiles_raw", "pIC50"}
missing_columns = required_columns.difference(df.columns)

# Eksik sütun varsa erken ve anlaşılır hata veriyoruz.
if missing_columns:
    raise KeyError(f"Eksik zorunlu sütunlar: {sorted(missing_columns)}")

# Başlangıç boyutunu yazdırıyoruz.
print("Başlangıç veri boyutu:", df.shape)

# İlk satırları gösteriyoruz.
display(df[["molecule_id", "smiles_raw", "pIC50"]].head(10))

assert len(df) > 0, "Giriş veri kümesi boş."

print("✅ CHECKPOINT 02.4: Veri başarıyla yüklendi ve zorunlu sütunlar doğrulandı.")

In [ ]:
# Bir SMILES'ı RDKit parent yapısına dönüştüren fonksiyonu tanımlıyoruz.
def standardize_to_parent(smiles):
    # Eksik değeri doğrudan geçersiz kabul ediyoruz.
    if pd.isna(smiles):
        return None

    # Girdiyi temiz bir metne çeviriyoruz.
    smiles = str(smiles).strip()

    # Boş metni geçersiz kabul ediyoruz.
    if not smiles:
        return None

    try:
        # SMILES metnini RDKit molekül nesnesine çeviriyoruz.
        mol = Chem.MolFromSmiles(smiles)

        # RDKit molekülü oluşturamazsa geçersiz kabul ediyoruz.
        if mol is None:
            return None

        # En anlamlı ana fragmanı seçiyor; tipik tuz ve karşı iyonları uzaklaştırıyoruz.
        parent = rdMolStandardize.FragmentParent(mol)

        # Standardizasyon sonrası molekülün kimyasal geçerliliğini kontrol ediyoruz.
        Chem.SanitizeMol(parent)

        # Parent yapıyı canonical ve stereokimyayı koruyan SMILES olarak yazıyoruz.
        return Chem.MolToSmiles(
            parent,
            canonical=True,
            isomericSmiles=True,
        )

    # Beklenmeyen kimyasal ayrıştırma hatalarında None döndürüyoruz.
    except Exception:
        return None


# Ham SMILES içinde nokta bulunmasını olası çoklu fragman/tuz göstergesi olarak işaretliyoruz.
df["had_multiple_fragments"] = (
    df["smiles_raw"].astype(str).str.contains(".", regex=False)
)

# Her SMILES için parent SMILES üretiyoruz.
df["parent_smiles"] = df["smiles_raw"].progress_apply(standardize_to_parent)

# Parent üretilemeyen satırları geçersiz olarak işaretliyoruz.
df["is_valid_smiles"] = df["parent_smiles"].notna()

# Sayısal özetleri yazdırıyoruz.
print("Toplam satır:", len(df))
print("Çoklu fragman/tuz göstergeli satır:", int(df["had_multiple_fragments"].sum()))
print("Geçersiz SMILES:", int((~df["is_valid_smiles"]).sum()))
print("Geçerli parent SMILES:", int(df["is_valid_smiles"].sum()))

# En az bir geçerli molekül kaldığını kontrol ediyoruz.
assert df["is_valid_smiles"].any(), "Hiçbir geçerli parent SMILES üretilemedi."

print("✅ CHECKPOINT 02.5: SMILES doğrulama ve parent yapı üretimi tamamlandı.")

In [ ]:
# Geçersiz SMILES satırlarını ayrı bir rapora alıyoruz.
invalid_smiles_df = df.loc[~df["is_valid_smiles"]].copy()

# Yalnızca geçerli parent yapılara sahip satırlarla devam ediyoruz.
valid_df = df.loc[df["is_valid_smiles"]].copy()

# Parent SMILES bazında kaç kayıt olduğunu hesaplıyoruz.
duplicate_counts = (
    valid_df.groupby("parent_smiles")
    .size()
    .sort_values(ascending=False)
    .rename("n_records")
    .reset_index()
)

# Birden fazla kaydı bulunan parent yapıları raporluyoruz.
duplicate_groups = duplicate_counts.query("n_records > 1").copy()

print("Benzersiz parent SMILES:", valid_df["parent_smiles"].nunique())
print("Tekrarlı parent grubu:", len(duplicate_groups))
display(duplicate_groups.head(20))

# Tekrarlı grup sayısının mantıklı bir tamsayı olduğunu kontrol ediyoruz.
assert len(duplicate_groups) >= 0

print("✅ CHECKPOINT 02.6: Parent SMILES tekrarları belirlendi.")

In [ ]:
# Bir sütundaki benzersiz metin değerlerini tek hücrede birleştiren yardımcı fonksiyonu tanımlıyoruz.
def join_unique_text(values):
    # Eksik olmayan değerleri metne çeviriyoruz.
    clean_values = [str(value) for value in values if pd.notna(value)]

    # Sıralı benzersiz değerleri noktalı virgülle birleştiriyoruz.
    return ";".join(sorted(set(clean_values)))


# Parent SMILES bazında aktivite ölçümlerini birleştiriyoruz.
dedup_df = (
    valid_df.groupby("parent_smiles", as_index=False)
    .agg(
        # Molekül kimliklerini izlenebilirlik için saklıyoruz.
        molecule_id=("molecule_id", join_unique_text),

        # Ham SMILES örneklerinden ilkini referans olarak saklıyoruz.
        smiles_raw_example=("smiles_raw", "first"),

        # pIC50 için uç değerlere daha dayanıklı medyanı kullanıyoruz.
        pIC50=("pIC50", "median"),

        # Alternatif karşılaştırma için ortalamayı da raporluyoruz.
        pIC50_mean=("pIC50", "mean"),

        # Ölçümler arası standart sapmayı raporluyoruz.
        pIC50_std=("pIC50", "std"),

        # En düşük ve en yüksek ölçümü saklıyoruz.
        pIC50_min=("pIC50", "min"),
        pIC50_max=("pIC50", "max"),

        # Her parent için kaç deneysel kayıt olduğunu saklıyoruz.
        n_records=("pIC50", "size"),

        # Kaynak kayıtlardan herhangi birinde tuz/çoklu fragman olup olmadığını saklıyoruz.
        had_multiple_fragments=("had_multiple_fragments", "max"),
    )
)

# Tek ölçümlü moleküllerde std NaN olacağı için 0 ile dolduruyoruz.
dedup_df["pIC50_std"] = dedup_df["pIC50_std"].fillna(0.0)

# Ölçümler arası farkı kolay yorumlamak için aralık hesaplıyoruz.
dedup_df["pIC50_range"] = dedup_df["pIC50_max"] - dedup_df["pIC50_min"]

# Yüksek deneysel değişkenliği işaretlemek için öğretici bir eşik kullanıyoruz.
dedup_df["activity_conflict_flag"] = dedup_df["pIC50_range"] > 1.0

print("Birleştirme öncesi satır:", len(valid_df))
print("Birleştirme sonrası benzersiz parent:", len(dedup_df))
print("Aktivite aralığı > 1 log birim olan parent:", int(dedup_df["activity_conflict_flag"].sum()))

# Birleştirme sonrasında parent SMILES tekrarının kalmadığını doğruluyoruz.
assert dedup_df["parent_smiles"].is_unique, "Dedup sonrasında tekrar kaldı."

print("✅ CHECKPOINT 02.7: Tekrarlı parent kayıtları medyan pIC50 ile birleştirildi.")

In [ ]:
# Bir parent SMILES için Lipinski özelliklerini hesaplayan fonksiyonu tanımlıyoruz.
def calculate_lipinski(smiles):
    # Parent SMILES'ı RDKit molekülüne dönüştürüyoruz.
    mol = Chem.MolFromSmiles(smiles)

    # Molekül oluşturulamazsa tüm alanları NaN döndürüyoruz.
    if mol is None:
        return pd.Series(
            {
                "MolWt": np.nan,
                "MolLogP": np.nan,
                "HBD": np.nan,
                "HBA": np.nan,
                "Lipinski_violations": np.nan,
                "passes_lipinski": False,
            }
        )

    # Molekül ağırlığını hesaplıyoruz.
    mol_wt = Descriptors.MolWt(mol)

    # Oktanol/su dağılım katsayısının hesaplanan logP değerini hesaplıyoruz.
    mol_logp = Crippen.MolLogP(mol)

    # Hidrojen bağı verici sayısını hesaplıyoruz.
    hbd = Lipinski.NumHDonors(mol)

    # Hidrojen bağı alıcı sayısını hesaplıyoruz.
    hba = Lipinski.NumHAcceptors(mol)

    # Her eşik aşımını bir ihlal olarak sayıyoruz.
    violations = int(mol_wt > 500)
    violations += int(mol_logp > 5)
    violations += int(hbd > 5)
    violations += int(hba > 10)

    # Hesaplanan değerleri tek satırlık Seri olarak döndürüyoruz.
    return pd.Series(
        {
            "MolWt": mol_wt,
            "MolLogP": mol_logp,
            "HBD": hbd,
            "HBA": hba,
            "Lipinski_violations": violations,
            "passes_lipinski": violations == 0,
        }
    )


# Her benzersiz parent yapı için Lipinski özelliklerini hesaplıyoruz.
lipinski_df = dedup_df["parent_smiles"].progress_apply(calculate_lipinski)

# Lipinski özelliklerini ana tabloyla yatay olarak birleştiriyoruz.
dedup_lipinski_df = pd.concat(
    [dedup_df.reset_index(drop=True), lipinski_df.reset_index(drop=True)],
    axis=1,
)

# Geçen ve kalan molekül sayılarını yazdırıyoruz.
print("Lipinski'den geçen:", int(dedup_lipinski_df["passes_lipinski"].sum()))
print("Lipinski'den kalan:", int((~dedup_lipinski_df["passes_lipinski"]).sum()))

# İhlal dağılımını gösteriyoruz.
display(
    dedup_lipinski_df["Lipinski_violations"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("ihlal_sayısı")
    .to_frame("molekül_sayısı")
)

assert dedup_lipinski_df["Lipinski_violations"].notna().all()

print("✅ CHECKPOINT 02.8: Lipinski özellikleri ve ihlal sayıları hesaplandı.")

In [ ]:
# Lipinski kurallarının tamamını geçen molekülleri modelleme setine alıyoruz.
filtered_df = dedup_lipinski_df.loc[
    dedup_lipinski_df["passes_lipinski"]
].copy()

# Lipinski nedeniyle elenenleri ayrı bir raporda saklıyoruz.
lipinski_rejected_df = dedup_lipinski_df.loc[
    ~dedup_lipinski_df["passes_lipinski"]
].copy()

# Çıktı yollarını tanımlıyoruz.
FILTERED_OUTPUT = CLEAN_DIR / "CHEMBL206_IC50_parent_dedup_Lipinski_filtered.csv"
ALL_STANDARDIZED_OUTPUT = CLEAN_DIR / "CHEMBL206_IC50_parent_dedup_with_Lipinski.csv"
INVALID_OUTPUT = CLEAN_DIR / "CHEMBL206_invalid_smiles.csv"
DUPLICATE_OUTPUT = CLEAN_DIR / "CHEMBL206_duplicate_parent_groups.csv"
REJECTED_OUTPUT = CLEAN_DIR / "CHEMBL206_Lipinski_rejected.csv"

# Tüm standardize edilmiş parent veri setini kaydediyoruz.
dedup_lipinski_df.to_csv(ALL_STANDARDIZED_OUTPUT, index=False)

# Modellemeye aktarılacak filtrelenmiş veri setini kaydediyoruz.
filtered_df.to_csv(FILTERED_OUTPUT, index=False)

# Kalite kontrol raporlarını kaydediyoruz.
invalid_smiles_df.to_csv(INVALID_OUTPUT, index=False)
duplicate_groups.to_csv(DUPLICATE_OUTPUT, index=False)
lipinski_rejected_df.to_csv(REJECTED_OUTPUT, index=False)

print("Modelleme girdi dosyası:", FILTERED_OUTPUT)
print("Tüm standardize parent dosyası:", ALL_STANDARDIZED_OUTPUT)
print("Geçersiz SMILES raporu:", INVALID_OUTPUT)
print("Tekrar grupları raporu:", DUPLICATE_OUTPUT)
print("Lipinski reddedilenler:", REJECTED_OUTPUT)

# Son veri setinin boş olmadığını kontrol ediyoruz.
assert len(filtered_df) > 0, "Lipinski filtresinden sonra veri kalmadı."

# Son veri setinde parent tekrarının kalmadığını kontrol ediyoruz.
assert filtered_df["parent_smiles"].is_unique

print("✅ CHECKPOINT 02.9: Temizlenmiş ve filtrelenmiş veriler başarıyla kaydedildi.")

In [ ]:
# Ders sırasında molekülleri görsel olarak kontrol etmek için ilk 12 yapıyı seçiyoruz.
preview_df = filtered_df.head(12).copy()

# SMILES'ları RDKit molekül nesnelerine dönüştürüyoruz.
preview_mols = [
    Chem.MolFromSmiles(smiles)
    for smiles in preview_df["parent_smiles"]
]

# Her molekülün altında pIC50 ve tekrar sayısını gösterecek açıklamalar hazırlıyoruz.
preview_legends = [
    f"pIC50={row.pIC50:.2f} | n={int(row.n_records)}"
    for row in preview_df.itertuples()
]

# Molekülleri 4 sütunlu bir ızgara halinde çiziyoruz.
grid_image = Draw.MolsToGridImage(
    preview_mols,
    molsPerRow=4,
    subImgSize=(280, 220),
    legends=preview_legends,
)

# Görseli notebook içinde gösteriyoruz.
display(grid_image)

print("✅ CHECKPOINT 02.10: Filtrelenmiş molekül ön izlemesi oluşturuldu.")

## Ders sonu değerlendirme soruları

1. `FragmentParent` işlemi ile yalnızca `"."` karakterine göre bölme arasındaki fark nedir?
2. Aynı parent yapı için pIC50 medyanı neden ortalamaya göre daha dayanıklı olabilir?
3. Lipinski filtresi “ilaç olur/olmaz” kararı verir mi, yoksa yalnızca bir ön eleme midir?
4. `activity_conflict_flag=True` olan moleküller modellemede nasıl ele alınabilir?

**Sonraki notebook:** Mordred 2D descriptor üretimi ve makine öğrenmesine uygun özellik matrisi.